# Guide 7: Pre-training: learning from a mountain of text

*Lifecycle* · [Read the guide](https://drvgautam.github.io/primer-on-aiengineering/pretraining/) · [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drvgautam/primer-on-aiengineering/blob/main/code/notebooks/07-pretraining.ipynb)

This notebook holds every code block from Guide 7, in the guide's order: 10 code cells, 9 of which print an output the guide quotes. Each cell is preceded by the guide paragraph that introduces it, so you can keep the guide open beside the notebook. The saved outputs are the guide's; `tools/verify_notebooks.py` checks they still match.

Run the setup cell first, then the cells in order: like the guide's blocks, they share variables.

**Sections of the guide**

- What pre-training is — no code
- A world bigger than one paragraph — 2 code cells
- From a token stream to batches — 2 code cells
- Better steps: Adam — 1 code cell
- Learning or memorizing? — 2 code cells
- Scale: parameters, data and compute — 1 code cell
- Making it fit — no code
- What comes out: a base model — 2 code cells

In [1]:
# Setup: run this cell first. It makes the repository's code/ folder the working directory, so
# every block below runs exactly as printed in the guide ("run from the repository root").
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/drvgautam/primer-on-aiengineering"
if "google.colab" in sys.modules and not any((p / "code" / "notebooks").is_dir() for p in [Path.cwd(), *Path.cwd().parents]):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/primer-on-aiengineering"], check=True)
    os.chdir("/content/primer-on-aiengineering")
    pins = [l.strip() for l in open("code/requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins,
                    "--extra-index-url", "https://download.pytorch.org/whl/cpu"], check=True)

here = Path.cwd().resolve()
code_dir = next(p / "code" for p in [here, *here.parents] if (p / "code" / "notebooks").is_dir())
os.chdir(code_dir)

# The guide's outputs were produced with these versions, on CPU. Others may differ in the last digits.
import numpy, tiktoken, torch
pins = dict(l.strip().split("==") for l in open("requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch"))
for name, mod in (("numpy", numpy), ("tiktoken", tiktoken), ("torch", torch)):
    have = mod.__version__.split("+")[0]
    note = "" if have == pins[name] else f"  <- the guide used {pins[name]}; outputs may differ slightly (on Colab, restart the runtime and rerun this cell)"
    print(f"{name:9} {have}{note}")
print("working directory:", Path(code_dir.parent.name) / "code")

# %%gpt2 marks blocks the guide shows but doesn't run, because they need GPT-2 small's 548 MB weights.
# To run them, download the weights as series/running-example.py explains, then set:
#     os.environ["PRIMER_RUN_GPT2"] = "1"
from IPython.core.magic import register_cell_magic

@register_cell_magic
def gpt2(line, cell):
    if os.environ.get("PRIMER_RUN_GPT2") == "1":
        get_ipython().run_cell(cell)
    else:
        print("Skipped: needs GPT-2 weights (see the setup cell to enable).")


numpy     2.2.6
tiktoken  0.14.0
torch     2.14.1
working directory: primer-on-aiengineering/code


## A world bigger than one paragraph

That's 6 × 4 × 4 × 6 = 576 possible sentences. The code holds back one fifth of them, 115 sentence types chosen at random, which never appear in training. They form the **held-out set**, also called the **validation set**: text kept back so you can test the model on something it has never seen. The code in this guide uses PyTorch 2.14.1, pinned in the series `requirements.txt` as of October 2026, and its blocks run in order and share variables.

In [2]:
import random
nouns  = ["cat", "dog", "rat", "fox", "owl", "hen"]
verbs  = ["sat", "ran", "hid", "lay"]
preps  = ["on", "by", "near", "under"]
places = ["mat", "log", "rug", "bed", "pot", "hut"]
types = [f"the {a} {b} {c} the {d}" for a in nouns for b in verbs for c in preps for d in places]
random.Random(0).shuffle(types)
val_types, train_types = types[:115], types[115:]          # one fifth held out

def corpus(types, n, seed):                                # n random sentences joined by " and "
    rng = random.Random(seed)
    return " and ".join(rng.choice(types) for _ in range(n))

print(len(types), "types:", len(train_types), "for training,", len(val_types), "held out")
print(corpus(train_types, 3, seed=1))

576 types: 461 for training, 115 held out
the owl ran on the rug and the hen ran on the hut and the fox ran by the hut


How low could any model's loss go on this text? Recall from Guide 6 that a loss computed with the natural log is measured in nats. Each sentence is four independent, uniform choices, and an even guess among 6 options costs $-\ln(1/6) = \ln 6$ nats. So picking one sentence carries $\ln 6 + \ln 4 + \ln 4 + \ln 6$ nats of uncertainty, and everything else ("the", the spaces, " and ") is fixed. Spread that over the average sentence's length and you get the lowest loss per character:

In [3]:
import math
nats = sum(math.log(len(words)) for words in (nouns, verbs, preps, places))   # uncertainty per sentence
chars = sum(len(t) for t in types) / len(types) + len(" and ")                # average characters per sentence
print(f"{nats:.3f} nats per sentence, {chars:.2f} characters, floor {nats / chars:.3f} nats per character")

6.356 nats per sentence, 28.25 characters, floor 0.225 nats per character


## From a token stream to batches

The toy's context is 32 characters. A corpus of 4,000 training sentences gives these counts:

> *`guides/tinygpt.py` holds the code built in Guide 5 ([notebook 05](05-transformer.ipynb)).*

In [4]:
import sys
sys.path.insert(0, "guides")                   # run from the repository root
import torch
import torch.nn.functional as F
from tinygpt import TinyGPT, vocab             # Guide 5's model and its 27 characters

def encode(text):
    return torch.tensor([vocab.index(c) for c in text])

def windows(ids, ctx, stride):                 # every ctx-long slice, one starting each stride characters
    inputs = ids[:-1].unfold(0, ctx, stride)   # (number of windows) × ctx
    targets = ids[1:].unfold(0, ctx, stride)   # the same windows, one step later
    return inputs, targets

train_text = corpus(train_types, 4000, seed=1)
train_ids = encode(train_text)
print(len(train_ids), "characters")
for stride in (1, 32):
    print("stride", stride, len(windows(train_ids, 32, stride)[0]), "windows")

112986 characters
stride 1 112954 windows
stride 32 3530 windows


Here's the training loop for the rest of the guide. Each step draws $B$ random windows at stride 1, so any starting character can be picked, computes their loss, and updates the parameters. Every so often it measures the loss on two fixed sets of windows at stride 32: the training text, and 300 sentences from the held-out types. Back-to-back windows count each character once and keep the check cheap, while training at stride 1 gets every possible starting point. The model is Guide 5's, with its position table stretched from 16 to 32 rows. Stretching it adds 16 rows of 32 numbers ($d_{\text{model}} = 32$), 512 parameters, to Guide 5's 27,739, for 28,251 in all. PyTorch calls minibatch gradient descent `SGD`, for stochastic gradient descent. To keep the whole guide under a minute on a laptop, every run here is 1,000 steps.

> *This output depends on a training run, so on your processor, or with a different number of threads, its last digits (and sometimes its text) can differ from the guide's.*

In [5]:
val_inputs, val_targets = windows(encode(corpus(val_types, 300, seed=99)), 32, 32)

def evaluate(model, inputs, targets):          # mean loss per character over all the windows
    with torch.no_grad():
        logits = model(inputs)                 # windows × n × |V|
        return F.cross_entropy(logits.reshape(-1, len(vocab)), targets.reshape(-1)).item()

def train(ids, method="adam", lr=0.003, steps=1000, B=32, seed=0, report=(500, 1000)):
    torch.manual_seed(seed)
    model = TinyGPT(n_max=32)                  # Guide 5's model, 32-character context
    inputs, targets = windows(ids, 32, 1)
    train_check = windows(ids, 32, 32)
    params = model.parameters()
    optimizer = torch.optim.Adam(params, lr=lr) if method == "adam" else torch.optim.SGD(params, lr=lr)
    generator = torch.Generator().manual_seed(seed)
    curve = {}
    for step in range(steps + 1):
        if step in report:
            curve[step] = (evaluate(model, *train_check), evaluate(model, val_inputs, val_targets))
        if step == steps:
            return model, curve
        batch = torch.randint(len(inputs), (B,), generator=generator)     # B random windows
        logits = model(inputs[batch])                                     # B × n × |V|
        loss = F.cross_entropy(logits.reshape(-1, len(vocab)), targets[batch].reshape(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step()     # clear old gradients (PyTorch adds to them), compute, update

model, curve = train(train_ids, report=(0, 50, 100, 250, 500, 750, 1000))
print(sum(param.numel() for param in model.parameters()), "parameters")
print(" step  training  validation")
for step, (train_loss, val_loss) in curve.items():
    print(f"{step:5}  {train_loss:8.3f}  {val_loss:10.3f}")

28251 parameters
 step  training  validation
    0     3.567       3.582
   50     1.329       1.341
  100     0.936       0.935
  250     0.405       0.400
  500     0.301       0.301
  750     0.287       0.291
 1000     0.281       0.287


## Better steps: Adam

The comparison below trains the same model on the same 4,000 sentences, with the same batches, using minibatch gradient descent (`SGD`) at two learning rates and Adam at 0.003, reusing the Adam run from above. The rates aren't comparable: gradient descent moves $\eta$ times the gradient, and these gradients are small, while Adam moves about $\eta$ per step whatever the gradient's size, so its useful rates are much smaller.

> *This output depends on a training run, so on your processor, or with a different number of threads, its last digits (and sometimes its text) can differ from the guide's.*

In [6]:
print("optimizer         learning rate  validation at 500  at 1,000")
for lr in (0.3, 1.0):
    _, c = train(train_ids, method="sgd", lr=lr)
    print(f"SGD (minibatch)   {lr:13}  {c[500][1]:17.3f}  {c[1000][1]:8.3f}")
print(f"Adam              {0.003:13}  {curve[500][1]:17.3f}  {curve[1000][1]:8.3f}")

optimizer         learning rate  validation at 500  at 1,000


SGD (minibatch)             0.3              0.827     0.369


SGD (minibatch)             1.0              0.324       nan
Adam                      0.003              0.301     0.287


## Learning or memorizing?

The tool is the validation set. Training loss can be driven down by memorizing the training text; loss on text the model has never seen can't. So train the same 28,251-parameter model for the same 1,000 Adam steps on just 10 sentences, and compare with the 4,000-sentence run:

> *This output depends on a training run, so on your processor, or with a different number of threads, its last digits (and sometimes its text) can differ from the guide's.*

In [7]:
print("sentences  training  validation")
_, c = train(encode(corpus(train_types, 10, seed=1)))
print(f"{10:9,}  {c[1000][0]:8.3f}  {c[1000][1]:10.3f}")
print(f"{4000:9,}  {curve[1000][0]:8.3f}  {curve[1000][1]:10.3f}")

sentences  training  validation


       10     0.112       1.656
    4,000     0.281       0.287


The most direct test is to let the model write. This samples 3,000 characters from the 4,000-sentence model with Guide 6's sampling at temperature 1, splits the text at " and ", and checks each complete sentence. So that every reader gets the same samples, it first loads that model as saved from the run above, in `guides/checkpoints/pretraining.pt`, as Guide 6 did with Guide 5's toy:

In [8]:
model = TinyGPT(n_max=32)                      # the 4,000-sentence run above, as saved
model.load_state_dict(torch.load("guides/checkpoints/pretraining.pt"))

def sample(model, n, seed):                    # Guide 6's loop: sample, append, repeat
    generator = torch.Generator().manual_seed(seed)
    ids = encode("the ")
    with torch.no_grad():
        while len(ids) < n:
            p = torch.softmax(model(ids[None, -32:])[0, -1], dim=-1)    # last 32 characters in, last position out
            ids = torch.cat([ids, torch.multinomial(p, 1, generator=generator)])
    return "".join(vocab[i] for i in ids)

sentences = sample(model, 3000, seed=0).split(" and ")[:-1]   # drop the unfinished last one
seen = set(train_text.split(" and "))
good = [s for s in sentences if s in types]
new = [s for s in good if s not in seen]
print(len(sentences), "sentences,", len(good), "grammatical,", len(new), "never in training",
      f"({len(set(new))} different),", sum(s in val_types for s in new), "of them held-out types")
print("new:", new[:3])
print("misses:", [s for s in sentences if s not in types][:4])

102 sentences, 85 grammatical, 13 never in training (13 different), 13 of them held-out types
new: ['the fox lay on the bed', 'the dog ran by the bed', 'the hen hid on the bed']
misses: ['the log', 'the  lay on the mat', 'the owl hid near the sog', 'the owl sat neat the pot']


## Scale: parameters, data and compute

floating-point operations, where $N$ is the number of parameters (Kaplan et al. count those outside the embedding tables; the toy's figure below uses all 28,251, a slight overcount), $D$ the number of training tokens, and $C$ the total compute ([Kaplan et al., 2020](https://arxiv.org/abs/2001.08361)). The rule counts only the parameters' multiplies and adds. Attention's scores add a cost that grows with context length; it's small at a thousand tokens, and the accordion below shows where it comes from. The toy saw 1,000 steps of 1,024 characters, and GPT-3 had 175 billion parameters trained on 300 billion tokens (Brown et al., 2020):

In [9]:
for name, N, D in [("toy", 28_251, 1000 * 32 * 32), ("GPT-3", 175e9, 300e9)]:
    print(f"{name:6} C = 6ND = {6 * N * D:.2e}")
C = 6 * 175e9 * 300e9
print(f"at 10^15 operations per second: {C / 1e15 / 3.15e7:.1f} years on one device, {C / 1e18 / 86400:.1f} days on 1,000")
for name, N, D in [("GPT-3", 175e9, 300e9), ("Chinchilla", 70e9, 1.4e12), ("LLaMA-7B", 7e9, 1e12)]:
    print(f"{name:10} {D / N:6.1f} tokens per parameter")

toy    C = 6ND = 1.74e+11
GPT-3  C = 6ND = 3.15e+23
at 10^15 operations per second: 10.0 years on one device, 3.6 days on 1,000
GPT-3         1.7 tokens per parameter
Chinchilla   20.0 tokens per parameter
LLaMA-7B    142.9 tokens per parameter


## What comes out: a base model

A base model continues text. Here is GPT-2 small, the series' model, continuing two prompts with Guide 6's greedy rule. Its weights are a 548 MB download, so the block is shown but not run here; `series/running-example.py` pins the revision and checks the file's hash.

In [10]:
%%gpt2
import runpy
script = runpy.run_path("series/running-example.py")         # needs the 548 MB weights file
for prompt in ["The capital of France is", "Q: What is the capital of France?\nA:"]:
    print(repr(script["greedy"](prompt, 25)))                 # 25 more tokens, most probable each time

Skipped: needs GPT-2 weights (see the setup cell to enable).


Back to the running example, "The cat sat on the". Recall from Guide 6 that GPT-2 small gives " mat" a probability of only about 0.004 there. As a loss, that's

In [11]:
%%gpt2
print(script["next_token_loss"](" mat"))                   # (loss in nats, perplexity)

Skipped: needs GPT-2 weights (see the setup cell to enable).


---

← [Guide 6: Next-token prediction](06-next-token.ipynb)

Found a cell that doesn't reproduce the guide? [Open an issue](https://github.com/drvgautam/primer-on-aiengineering/issues/new?template=does-not-reproduce.yml).